In [7]:
pip install transformers torch pandas

   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 2.9/241.4 MB 13.9 MB/s eta 0:00:18
   - -------------------------------------- 6.8/241.4 MB 16.8 MB/s eta 0:00:14
   - -------------------------------------- 11.0/241.4 MB 18.1 MB/s eta 0:00:13
   -- ------------------------------------- 16.0/241.4 MB 20.1 MB/s eta 0:00:12
   --- ------------------------------------ 21.8/241.4 MB 21.5 MB/s eta 0:00:11
   ---- ----------------------------------- 27.8/241.4 MB 22.6 MB/s eta 0:00:10
   ----- ---------------------------------- 33.3/241.4 MB 23.0 MB/s eta 0:00:10
   ------ --------------------------------- 37.7/241.4 MB 23.5 MB/s eta 0:00:09
   ------- -------------------------------- 42.5/241.4 MB 23.3 MB/s eta 0:00:09
   ------- -------------------------------- 47.4/241.4 MB 23.2 MB/s eta 0:00:09
   -------- ------------------------------- 52.7/241.4 MB 23.1 MB/s eta 0:00:09
   --------- ------------------------------ 58.7/24


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\geova\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
from transformers import pipeline
import pandas as pd

# 1. Carregando os modelos
print("Carregando modelos. Isso pode levar alguns minutos...")

# Pipeline para Q&A
qa_pipeline = pipeline("question-answering", model="neuralmind/bert-base-portuguese-cased")

# Pipeline para Análise de Sentimento
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# Pipeline para Tradução (Português -> Inglês)
translation_pipeline = pipeline("translation_pt_to_en", model="Helsinki-NLP/opus-mt-pt-en")

# Pipeline para Classificação de Texto
classifier_pipeline = pipeline("zero-shot-classification", model="facebook/mbart-large-50-finetuned-conll03-en")

print("Modelos carregados com sucesso!")

ValueError: Name tf.RaggedTensorSpec has already been registered for class tensorflow.python.ops.ragged.ragged_tensor.RaggedTensorSpec.

In [ ]:
# Contexto para o chatbot
contexto_universidade = """
O Serviço Nacional de Aprendizagem Comercial (Senac) é uma instituição brasileira de educação criada em 10 de janeiro de 1946 através do decreto-lei 8.621.[3] É uma entidade privada com fins públicos que recebe contribuição compulsória das empresas do comércio e de atividades assemelhadas. A nível nacional é administrado pela Confederação Nacional do Comércio.
Os primeiro cursos ofertados pelo SENAC foram em 1947, foi o curso de Praticante de Comércio e o curso de Praticante de Escritório, destinado a jovens entre 14 e 18 anos. Já para maiores de 18 anos, foram disponibilizados os cursos de Balconista de Tecidos, Calçados e Ferragens, Arquivista e Caixa-Tesoureiro.
"""

# Função para processar uma única pergunta
def processar_pergunta(pergunta, contexto):
    try:
        # 1. Resposta Q&A
        resposta_qa = qa_pipeline(question=pergunta, context=contexto)

        # 2. Traduzir a pergunta para inglês
        pergunta_traduzida = translation_pipeline(pergunta)[0]['translation_text']

        # 3. Análise de Sentimento
        sentimento = sentiment_pipeline(pergunta_traduzida)

        # 4. Classificar a pergunta
        categorias = ['cursos', 'instituição']
        classificacao = classifier_pipeline(pergunta, candidate_labels=categorias)

        # Imprimir os resultados de forma clara
        print("\n--- Resultados ---")
        print(f"**Pergunta Original:** {pergunta}")
        print(f"**Resposta Q&A:** {resposta_qa['answer']}")
        print(f"**Sentimento:** {sentimento[0]['label']} (score: {sentimento[0]['score']:.2f})")
        print(f"**Pergunta Traduzida (en):** {pergunta_traduzida}")
        print(f"**Classificação:** {classificacao['labels'][0]}")
        print("------------------\n")

    except Exception as e:
        print(f"Ocorreu um erro ao processar a pergunta: {e}")


# Loop principal do chatbot
s = ''
print("Olá! Eu sou o seu chatbot. Digite sua pergunta ou 'sair' para encerrar.")

while s != 'sair':
    s = input("Você: ")
    if s.lower() == 'sair':
        print("Chatbot encerrado. Até mais!")
        break
    
    processar_pergunta(s, contexto_universidade)